In [12]:
from pathlib import Path
import contextlib, io

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import geopandas as gpd
import rasterio.features
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from downscaling.read_process_grid_data import calculate_resolution
from downscaling.downscaling import aggregate_urban_values
from downscaling.process_urban_grid_emissions import precompute_urban_masks

In [13]:
# Read data: GADM raster with IMAGE regions, gridded population and downscaled and harmonised emissions data (IAM pathways), and city pledges data
# Calculate: For each region, select the cells that start in 2020 with the same per capita emissions (using a threshold)
# Plot: plot the per capita emissions over time, and compare with the cities pledges data

# Structure of the code:
# 1. settings
# 2. definitions of functions
# 3. Calculations and plotting for each selected region

In [ ]:
# settings

calculate_IMAGE_urban_pathways = False

project_dir = Path.cwd()
(project_dir / "data/check/urban_comparison").mkdir(parents=True, exist_ok=True)
urban_comparison_dir = project_dir / "data/check/urban_comparison"
urban_comparison_dir.mkdir(exist_ok=True)
dir_IMAGE = project_dir / "data/input/Models/IMAGE"
print(f"IMAGE directory: {dir_IMAGE}")

file_IMAGE_regions = "image_region_numbers.csv"
path_IMAGE_regions = dir_IMAGE / file_IMAGE_regions
# read IMAGE region numbers to ISO codes
df_IMAGE_regions = pd.read_csv(path_IMAGE_regions, sep=",") # ISO3 --> region_number
df_IMAGE_regions = df_IMAGE_regions.rename(columns={"IMAGE number": "region_number", "IMAGE region": "iso"})
df_IMAGE_regions["region_number"] = df_IMAGE_regions["region_number"].astype(int)

# change IMGAE codes to ISO3 codes for some countries
IMAGE_recion_code_to_iso = {"INDIA": "IND",
                            "INDO": "IDN",
                            "SAF": "ZAF",
                            "JAP": "JPN"}
df_IMAGE_regions["iso"] = df_IMAGE_regions["iso"].replace(IMAGE_recion_code_to_iso)
region_to_iso = dict(zip(df_IMAGE_regions['region_number'], df_IMAGE_regions['iso']))
iso_to_region = dict(zip(df_IMAGE_regions['iso'], df_IMAGE_regions['region_number']))
print(f"\nRegion to ISO mapping:\n{region_to_iso}")
print(f"\nISO to Region mapping:\n{iso_to_region}")
region_to_iso = dict(zip(df_IMAGE_regions['region_number'], df_IMAGE_regions['iso']))
iso_to_region = dict(zip(df_IMAGE_regions['iso'], df_IMAGE_regions['region_number']))


varname_em = "Emissions_CO2_Excl_shipping_aviation_AFOLU"
varname_em_per_capita = "Emissions_per_capita"

scenarios = ["ELV-SSP2-CP", "ELV-SSP2-1150F"]
IMAGE_scenarios = ["IMAGE_" + s for s in scenarios]
print(IMAGE_scenarios)

IMAGE directory: k:\PythonWork\downscaling\Kaya_downscaling\data\input\Models\IMAGE
['IMAGE_ELV-SSP2-CP', 'IMAGE_ELV-SSP2-1150F']


# Functions

In [15]:
def _plot_hist(project_dir: Path, region_number: int, ds_em_per_capita_region_2020: xr.DataArray, per_capita_threshold: float, tolerance: float):
    # plot the distribution of per capita emissions in 2020 for the region
    perc_plot = 0.90
    high_perc = ds_em_per_capita_region_2020.quantile(perc_plot).values
    high_perc = float(high_perc) if isinstance(high_perc, np.ndarray) else high_perc

    data = ds_em_per_capita_region_2020.values.ravel() # takes an array of any shape and returns a 1D version of it, with all the elements laid out in a single flat sequence
    data = data[np.isfinite(data)]

    plt.figure(figsize=(10, 6))
    plt.hist(data, bins=50, range=(0, high_perc), color="steelblue", edgecolor="black")
    plt.axvline(per_capita_threshold, color="red", linestyle="dashed", linewidth=2, label=f"Threshold: {per_capita_threshold:.2f}")
    plt.axvline(per_capita_threshold - tolerance, color="orange", linestyle="dashed", linewidth=2, label=f"Lower bound: {per_capita_threshold - tolerance:.2f}")
    plt.axvline(per_capita_threshold + tolerance, color="orange", linestyle="dashed", linewidth=2, label=f"Upper bound: {per_capita_threshold + tolerance:.2f}")
    plt.title(f"Distribution of per capita emissions in 2020 for region {region_number}, cut-off at {perc_plot*100:.0f}%-percentile")
    plt.xlabel("Emissions per capita")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(project_dir / "figures" / f"hist_emissions_per_capita_region_{region_number}.png", dpi=200)
    plt.close()


In [16]:
def _plot_selected_cells(project_dir: Path, xr_em_per_capita_selected: xr.Dataset, varname_em_per_capita: str, region_number: int):
    fig, ax = plt.subplots(figsize=(12, 6), subplot_kw={"projection": ccrs.PlateCarree()})

    da = xr_em_per_capita_selected[varname_em_per_capita].sel(time=2020)
    if "lon" in da.dims and "lat" in da.dims:
        da = da.rename({"lon": "x", "lat": "y"})

    values = da.values                 # 2D array (y, x)
    mask = np.isfinite(values)         # True only for the selected (non-NaN) cells
    yi, xi = np.where(mask)

    x_coords = da["x"].values[xi]
    y_coords = da["y"].values[yi]
    point_values = values[mask]

    sc = ax.scatter(x_coords, y_coords, c=point_values, cmap="viridis", s=2, transform=ccrs.PlateCarree())                          # s = point size; increase to enlarge
    fig.colorbar(sc, ax=ax, label="Emissions per capita")

    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linewidth=0.5)
    ax.set_title("Selected cells: per capita emissions near threshold, 2020")

    save_path = project_dir / "figures" / f"map_emissions_per_capita_selected_region_{region_number}.png"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

In [17]:
def _print_region_stats(xr: xr.DataArray):

    print(f"  Min: {xr.min().values:.4f}")
    print(f"  Max: {xr.max().values:.4f}")
    print(f"  Mean: {xr.mean().values:.4f}")
    print(f"  Median: {xr.median().values:.4f}")
    print(f"  5%-percentile: {xr.quantile(0.05).values:.4f}")
    print(f"  95%-percentile: {xr.quantile(0.95).values:.4f}")

In [18]:
def _calc_avg_emissions_per_capita_selected(xr_em_per_capita_selected: xr.Dataset, varname_em_per_capita: str,
                                            xr_population: xr.Dataset, varname_population: str,
                                            arc_minutes: float) -> pd.DataFrame:

    # Align population data to the emissions dataset using nearest neighbor reindexing
    xr_population_aligned = xr_population.reindex_like(xr_em_per_capita_selected, method="nearest", tolerance=arc_minutes / 60 / 2)
    emissions_absolute = xr_em_per_capita_selected[varname_em_per_capita] * xr_population_aligned[varname_population]
    emissions_absolute = emissions_absolute.rename("emissions_absolute")

    # Calculate total emissions and population per region, andper capita emissions
    emissions_per_region = (emissions_absolute.groupby(xr_em_per_capita_selected["region_number"])
                            .sum())
    population_per_region = (xr_population_aligned.where(~xr_em_per_capita_selected[varname_em_per_capita].isnull())
                            .groupby(xr_em_per_capita_selected["region_number"])
                            .sum())
    region_per_capita = (emissions_per_region / population_per_region[varname_population]).rename("region_per_capita")

    # add to dataframe
    df_region_totals = (emissions_per_region.to_dataframe(name="total_emissions")
                        .reset_index()
                        .merge(population_per_region[varname_population].to_dataframe(name="total_population").reset_index(), on="region_number")
                        .merge(region_per_capita.to_dataframe().reset_index(), on="region_number"))

    return df_region_totals

In [19]:
from typing import Tuple

from duckdb import df

def select_cells_based_on_per_capita(region_number: int,
                                     xr_em_per_capita: xr.Dataset, varname_em_per_capita: str,
                                     xr_population: xr.Dataset, varname_population: str,
                                     only_urban: bool = False,
                                     da_urban_masks: xr.DataArray|None=None,
                                     per_capita_threshold: float=2.0,
                                     perc_select: float = 0.05, arc_minutes: float = 0.5) -> Tuple[pd.DataFrame, xr.Dataset]:
    # pre: xr_em_per_capita has at least two variables: <varname> and "region_number"
    '''
    Select cells from the emissions per capita dataset that are within a certain percentage of a given threshold
    '''

    # 0. Restrict to the region of interest — single source of truth for all later steps
    ds_em_per_capita_region = xr_em_per_capita.where(xr_em_per_capita["region_number"] == region_number)

    # 1. selection cells based on perc_select, per_capita_threshold, and values of cells in the year 2020
    print(f"1. Selecting cells with per capita emissions within {perc_select*100}% of {per_capita_threshold} in 2020...")
    tolerance = perc_select * per_capita_threshold
    ds_em_per_capita_region_2020 = ds_em_per_capita_region[varname_em_per_capita].sel(time=2020).drop_vars("time")

    # print min, max, mean, median of ds_em_per_capita_2020
    print(f"\nIn select_cells_based_on_per_capita for region_number: {region_number}")
    print(f"Per capita emissions in 2020 for region without selection based on threshold {region_number}:")
    _print_region_stats(ds_em_per_capita_region_2020)

    # plot the distribution of per capita emissions in 2020 for the region
    _plot_hist(project_dir, region_number, ds_em_per_capita_region_2020, per_capita_threshold, tolerance)

    # restrict to urban cells using the precomputed mask for the requested year
    mask = ((ds_em_per_capita_region_2020 >= per_capita_threshold - tolerance)
             & (ds_em_per_capita_region_2020 <= per_capita_threshold + tolerance)).rename("is_match")

    # restrict to urban cells using the precomputed mask for the requested year
    if only_urban and da_urban_masks is not None:
        is_urban = da_urban_masks.sel(year=2020).astype(bool)
        mask = (mask & is_urban).rename("is_match")

    match_count = int(mask.sum())
    print(f"\nFound {match_count} matching cells in region {region_number}.")
    df_matches = pd.DataFrame({"region_number": [region_number], "match_count": [match_count]})

    # 2. based on selection using mask, cells for all years are selected
    print(f"\n2. Selecting cells for all years based on the mask...")
    ds_em_per_capita_selected = ds_em_per_capita_region.where(mask)
    # print min, max, mean, median of ds_em_per_capita_2020
    ds_em_per_capita_selected_2020 = ds_em_per_capita_selected[varname_em_per_capita].sel(time=2020).drop_vars("time")
    print(f"Per capita emissions in 2020 for region with selection based on threshold {region_number}:")
    _print_region_stats(ds_em_per_capita_selected_2020)
    print(f"Per capita emissions in 2020 for region {region_number}:")

    total_cells = int((xr_em_per_capita["region_number"] == region_number).sum())

    df_matches["total_cells"] = total_cells
    df_matches["pct_matching"] = (100 * df_matches["match_count"] / df_matches["total_cells"]).round(1)
    print(f"\n{df_matches.to_string(index=False)}")

    # 3. plot selected cells for region on a map
    print(f"\n3. Plotting selected cells on a map...")
    _plot_selected_cells(project_dir, ds_em_per_capita_selected, varname_em_per_capita, region_number)

    # 4. calculate average emissions per capita for selected cells
    print(f"\n4. Calculating average emissions per capita for selected cells...")
    ds_em_per_capita_selected_2020 = ds_em_per_capita_selected[varname_em_per_capita].sel(time=2020).drop_vars("time")
    print(f"Per capita emissions in 2020 for region {region_number}:")

    df_region_total = _calc_avg_emissions_per_capita_selected(ds_em_per_capita_selected, varname_em_per_capita, xr_population, varname_population, arc_minutes)

    return df_region_total, ds_em_per_capita_selected

In [ ]:
def _plot_pathways(project_dir: Path, region_number: int, ds_em_per_capita: dict[str, xr.Dataset], ds_urban_em_per_capita_selected: dict[str, xr.Dataset], varname_em_per_capita: str,
                   df_IMAGE_pathways: dict[str, pd.DataFrame], df_IMAGE_urban_pathways: dict[str, pd.DataFrame],
                   df_pledge_cities_country: pd.DataFrame):


    '''
    Plot the per capita emissions over time for a specific region, including individual cell data, min-max range, and median.
    '''

    region_data: dict[str, xr.DataArray] = {}
    urban_region_data: dict[str, xr.DataArray] = {}
    df_region: dict[str, pd.DataFrame] = {}
    df_urban_region: dict[str, pd.DataFrame] = {}
    df_IMAGE_selected_pathways: dict[str, pd.DataFrame] = {}
    df_IMAGE_selected_urban_pathways: dict[str, pd.DataFrame] = {}
    for scenario in IMAGE_scenarios:
        region_data[scenario] = ds_em_per_capita[scenario][varname_em_per_capita]
        #urban_region_data[scenario] = ds_urban_em_per_capita_selected[scenario][varname_em_per_capita]
        # check if region_number in region_data_2020 is the same as the region_number passed to the function
        if region_number not in region_data[scenario]["region_number"].values:
            print(f"Warning: region_number {region_number} not found in region_data_2020. Available region_numbers: {region_data[scenario]['region_number'].values}")
            return
        df_region[scenario] = (region_data[scenario].to_dataframe(name=varname_em_per_capita)
                    .dropna(subset=[varname_em_per_capita])
                    .reset_index())
        #df_urban_region[scenario] = (urban_region_data[scenario].to_dataframe(name=varname_em_per_capita)
        #    .dropna(subset=[varname_em_per_capita])
        #    .reset_index())

        df_IMAGE_selected_pathways[scenario] = (df_region[scenario].groupby("time")[varname_em_per_capita]
                .agg(["min", "max", "median"])
                .reset_index())

        # df_IMAGE_selected_urban_pathways[scenario] = (df_urban_region[scenario].groupby("time")[varname_em_per_capita]
        #         .agg(["min", "max", "median"])
        #         .reset_index())

    # plot pathways
    fig, axs = plt.subplots(figsize=(10, 6), ncols=2, nrows=2)

    axs[0,0].plot(df_pledge_cities_country["year"], df_pledge_cities_country["emissions_per_cap"], linestyle="dotted", color="seagreen", marker=None, label="Cities Pledges")
    # TO DO : This needs to be change to urban pathways
    axs[0,0].plot(df_IMAGE_pathways[IMAGE_scenarios[0]]["year"], df_IMAGE_pathways[IMAGE_scenarios[0]]["value"], linestyle="solid", color="darkorange", marker=None, label="CP: Full IAM pathway")
    axs[0,0].plot(df_IMAGE_pathways[IMAGE_scenarios[1]]["year"], df_IMAGE_pathways[IMAGE_scenarios[1]]["value"], linestyle="solid", color="seagreen", marker=None, label="2C: Full IAM pathway")
    axs[0,0].set_xlabel("Year")
    axs[0,0].set_ylabel("Emissions per capita")
    axs[0,0].set_xlim(2020, 2050)
    axs[0,0].set_title(f"Per capita emission pathways over time\nregion {region_number}")
    axs[0,0].legend()

    axs[0,1].plot(df_pledge_cities_country["year"], df_pledge_cities_country["emissions_per_cap"], linestyle="dotted", color="seagreen", marker=None, label="Cities Pledges")
    #axs[0,1].plot(df_IMAGE_urban_pathways[IMAGE_scenarios[0]]["year"], df_IMAGE_urban_pathways[IMAGE_scenarios[0]]["value"], linestyle="solid", color="darkred", marker=None, label="CP: Full IAM pathway")
    #axs[0,1].plot(df_IMAGE_urban_pathways[IMAGE_scenarios[1]]["year"], df_IMAGE_urban_pathways[IMAGE_scenarios[1]]["value"], linestyle="solid", color="salmon", marker=None, label="2C: Full IAM pathway")
    axs[0,0].plot(df_IMAGE_pathways[IMAGE_scenarios[0]]["year"], df_IMAGE_pathways[IMAGE_scenarios[0]]["value"], linestyle="solid", color="darkorange", marker=None, label="CP: Full IAM pathway")
    axs[0,0].plot(df_IMAGE_pathways[IMAGE_scenarios[1]]["year"], df_IMAGE_pathways[IMAGE_scenarios[1]]["value"], linestyle="solid", color="seagreen", marker=None, label="2C: Full IAM pathway")
    # TO DO : This needs to be change to urban pathways and urban selected pathways
    axs[0,1].plot(df_IMAGE_selected_pathways[IMAGE_scenarios[0]]["time"], df_IMAGE_selected_pathways[IMAGE_scenarios[0]]["median"], linestyle="dashed", color="darkblue", marker=None, label="CP: Median (selected) IAM pathway")
    axs[0,1].plot(df_IMAGE_selected_pathways[IMAGE_scenarios[1]]["time"], df_IMAGE_selected_pathways[IMAGE_scenarios[1]]["median"], linestyle="dashed", color="lightblue", marker=None, label="2C: Median (selected) IAM pathway")
    #axs[0].plot(df_IMAGE_selected_urban_pathways[IMAGE_scenarios[0]]["time"], df_IMAGE_selected_urban_pathways[IMAGE_scenarios[0]]["median"], linestyle="dashed", color="gold", marker=None, label="CP: Median (selected) urban IAM pathway")
    #axs[0].plot(df_IMAGE_selected_urban_pathways[IMAGE_scenarios[1]]["time"], df_IMAGE_selected_urban_pathways[IMAGE_scenarios[1]]["median"], linestyle="dashed", color="palegoldenrod", marker=None, label="2C: Median (selected) urban IAM pathway")
    axs[0,1].set_xlabel("Year")
    axs[0,1].set_ylabel("Emissions per capita")
    axs[0,1].set_xlim(2020, 2050)
    axs[0,1].set_title(f"Per capita emission pathways over time\nregion {region_number}")
    axs[0,1].legend()

    # TO DO: make a comparison of IMAGE pathway and IMAGE urban pathway
    # axs[0,2]

    axs[1,0].plot(df_IMAGE_selected_pathways[IMAGE_scenarios[0]]["time"], df_IMAGE_selected_pathways[IMAGE_scenarios[0]]["median"], linestyle="solid", color="darkblue", marker="o", label="CP: Median (selected) IAM pathway")
    axs[1,0].scatter(df_region[IMAGE_scenarios[0]]["time"], df_region[IMAGE_scenarios[0]][varname_em_per_capita], alpha=0.2, s=10, color="steelblue", label="Individual cells")
    axs[1,0].fill_between(df_IMAGE_selected_pathways[IMAGE_scenarios[0]]["time"], df_IMAGE_selected_pathways[IMAGE_scenarios[0]]["min"], df_IMAGE_selected_pathways[IMAGE_scenarios[0]]["max"], alpha=0.15, color="steelblue", label="Min–max range")
    axs[1,0].set_title(f"Full pixel range\n{IMAGE_scenarios[0]}")
    axs[1,0].set_xlim(2020, 2050)
    axs[1,0].legend()

    axs[1,1].plot(df_IMAGE_selected_pathways[IMAGE_scenarios[1]]["time"], df_IMAGE_selected_pathways[IMAGE_scenarios[1]]["median"], linestyle="solid", color="lightblue", marker="o", label="2C: Median (selected) urban IAM pathway")
    axs[1,1].scatter(df_region[IMAGE_scenarios[1]]["time"], df_region[IMAGE_scenarios[1]][varname_em_per_capita], alpha=0.2, s=10, color="steelblue", label="Individual cells")
    axs[1,1].fill_between(df_IMAGE_selected_pathways[IMAGE_scenarios[1]]["time"], df_IMAGE_selected_pathways[IMAGE_scenarios[1]]["min"], df_IMAGE_selected_pathways[IMAGE_scenarios[1]]["max"], alpha=0.15, color="steelblue", label="Min–max range")
    axs[1,1].set_title(f"Full pixel range\n{IMAGE_scenarios[1]}")
    axs[1,1].set_xlim(2020, 2050)
    axs[1,1].legend()

    plt.tight_layout()
    save_path = project_dir / "figures" / f"emissions_per_capita_selected_region_{region_number}.png"
    plt.savefig(save_path, dpi=200, bbox_inches="tight")

# Read in data

## Region raster GADM with IMAGE regions

In [21]:
# read in IMAGE_GADM_regions_raster_6_00_arcmin.nc

dir_GADM_regions_raster = project_dir / "data/processed/GADM"
file_GADM_regions_raster =  "IMAGE_GADM_regions_raster_6_00_arcmin.nc"
path_GADM_regions_raster = dir_GADM_regions_raster / file_GADM_regions_raster
for item in dir_GADM_regions_raster.rglob("*.nc"):
    print(item)

print(f"\nFound GADM regions raster file: {path_GADM_regions_raster}")
with xr.open_dataset(path_GADM_regions_raster) as GADM_regions_raster:
    arc_seconds, arc_minutes, arc_degrees = calculate_resolution(GADM_regions_raster["region_number"])
    print(f"\nResolution of GADM regions raster: {arc_seconds} arcseconds, {arc_minutes} arcminutes, {arc_degrees} degrees" )
    print(f"\nGADM_regions_raster: {GADM_regions_raster}")
    # sort and print unique region numbers
    print(f"\nUnique region numbers in GADM regions raster: {np.unique(GADM_regions_raster['region_number'].values)}")
    # sort and print unique country codes
    print(f"\nUnique ISO country codes (country_id_GADM) in GADM regions raster: {np.unique(GADM_regions_raster['country_id_GADM'].values)}")

    id_to_iso_GADM = pd.read_csv(project_dir /"data/processed/GADM/id_to_iso_mapping.csv", sep=";")
    print(f"\nID to ISO mapping:\n{id_to_iso_GADM.head()}")
    print(GADM_regions_raster)

2026-08-28 17:22:16 - debug - read_process_grid_data:calculate_resolution:376 (from 319006044:<module>:11) - INFO - Calculate resolution for DataArray: <xarray.DataArray 'region_number' (y: 1800, x: 3600)> Size: 52MB
[6480000 values with dtype=int64]
Coordinates:
  * x        (x) float64 29kB -179.9 -179.8 -179.8 -179.6 ... 179.8 179.9 180.0
  * y        (y) float64 14kB 89.95 89.85 89.75 89.65 ... -89.75 -89.85 -89.95
2026-08-28 17:22:16 - debug - read_process_grid_data:calculate_resolution:377 (from 319006044:<module>:11) - INFO - CRS: None


k:\PythonWork\downscaling\Kaya_downscaling\data\processed\GADM\IMAGE_GADM_regions_raster_0_50_arcmin.nc
k:\PythonWork\downscaling\Kaya_downscaling\data\processed\GADM\IMAGE_GADM_regions_raster_6_00_arcmin.nc

Found GADM regions raster file: k:\PythonWork\downscaling\Kaya_downscaling\data\processed\GADM\IMAGE_GADM_regions_raster_6_00_arcmin.nc

Resolution of GADM regions raster: 360.0 arcseconds, 6.0 arcminutes, 0.1 degrees

GADM_regions_raster: <xarray.Dataset> Size: 104MB
Dimensions:          (y: 1800, x: 3600)
Coordinates:
  * x                (x) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.9 180.0
  * y                (y) float64 14kB 89.95 89.85 89.75 ... -89.75 -89.85 -89.95
Data variables:
    spatial_ref      int64 8B ...
    country_id_GADM  (y, x) float64 52MB ...
    region_number    (y, x) int64 52MB ...

Unique region numbers in GADM regions raster: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27]

Unique ISO country codes (country_

## IMAGE (IAM) pathways for current policies and 2C scenarios

In [22]:
# read scenario templates for IMAGE
vars_CO2_cities = ["Emissions|CO2|Energy|Supply", "Emissions|CO2|Energy|Demand",
                   "Emissions|CO2|Industrial Processes", "Emissions|CO2|Energy|Demand|Transportation|Domestic Aviation", "Emissions|CO2|Energy|Demand|Transportation|Domestic Shipping",
                   "Emissions|CO2|Energy|Demand|Bunkers|International Aviation",
                   "Population"]
dir_IMAGE_pathways = dir_IMAGE / "SSP"
df_IMAGE_pathways: dict[str, pd.DataFrame] = {}
df_IMAGE_pathways[IMAGE_scenarios[0]] = pd.read_excel(dir_IMAGE_pathways / f"{scenarios[0]}.xlsx", sheet_name="data")
df_IMAGE_pathways[IMAGE_scenarios[1]] = pd.read_excel(dir_IMAGE_pathways / f"{scenarios[1]}.xlsx", sheet_name="data")

for IMAGE_scenario in IMAGE_scenarios:
    unit_em = df_IMAGE_pathways[IMAGE_scenario][df_IMAGE_pathways[IMAGE_scenario]["Variable"] == "Emissions|CO2|Energy|Supply"]["Unit"].values[0]
    unit_pop = df_IMAGE_pathways[IMAGE_scenario][df_IMAGE_pathways[IMAGE_scenario]["Variable"] == "Population"]["Unit"].values[0]
    df_IMAGE_pathways[IMAGE_scenario].drop(["Unit"], axis=1, inplace=True)
    print(f"\nUnits for {IMAGE_scenario}: Emissions: {unit_em}, Population: {unit_pop}")
    df_IMAGE_pathways[IMAGE_scenario] = df_IMAGE_pathways[IMAGE_scenario][df_IMAGE_pathways[IMAGE_scenario]["Variable"].isin(vars_CO2_cities)]
    df_IMAGE_pathways[IMAGE_scenario] = df_IMAGE_pathways[IMAGE_scenario].melt(id_vars=["Model", "Scenario", "Region", "Variable"], var_name="year", value_name="value")
    df_IMAGE_pathways[IMAGE_scenario]["year"] = df_IMAGE_pathways[IMAGE_scenario]["year"].astype(int)
    df_IMAGE_pathways[IMAGE_scenario] = df_IMAGE_pathways[IMAGE_scenario].pivot(index=["Model", "Scenario", "Region", "year"], columns="Variable", values="value").reset_index()
    df_IMAGE_pathways[IMAGE_scenario]["Emissions_CO2_Excl_shipping_aviation_AFOLU"] = (df_IMAGE_pathways[IMAGE_scenario]["Emissions|CO2|Energy|Supply"] +
                                                                                       df_IMAGE_pathways[IMAGE_scenario]["Emissions|CO2|Energy|Demand"] +
                                                                                       df_IMAGE_pathways[IMAGE_scenario]["Emissions|CO2|Industrial Processes"] -
                                                                                       df_IMAGE_pathways[IMAGE_scenario]["Emissions|CO2|Energy|Demand|Transportation|Domestic Aviation"] -
                                                                                       df_IMAGE_pathways[IMAGE_scenario]["Emissions|CO2|Energy|Demand|Transportation|Domestic Shipping"]-
                                                                                       df_IMAGE_pathways[IMAGE_scenario]["Emissions|CO2|Energy|Demand|Bunkers|International Aviation"]) #-
    df_IMAGE_pathways[IMAGE_scenario]["emissions_per_capita"] = df_IMAGE_pathways[IMAGE_scenario]["Emissions_CO2_Excl_shipping_aviation_AFOLU"] / df_IMAGE_pathways[IMAGE_scenario]["Population"]
    df_IMAGE_pathways[IMAGE_scenario]["Unit"] = f"{unit_em}/{unit_pop}"
    print(f"\nUnits for {IMAGE_scenario}: Emissions per capita: {df_IMAGE_pathways[IMAGE_scenario]['Unit'].values[0]}")
    df_IMAGE_pathways[IMAGE_scenario] = df_IMAGE_pathways[IMAGE_scenario].melt(id_vars=["Model", "Scenario", "Region", "year", "Unit"], value_vars=["emissions_per_capita"], var_name="Variable", value_name="value")
    df_IMAGE_pathways[IMAGE_scenario] = df_IMAGE_pathways[IMAGE_scenario][df_IMAGE_pathways[IMAGE_scenario]["Variable"] == "emissions_per_capita"]
    df_IMAGE_pathways[IMAGE_scenario]["iso"] = df_IMAGE_pathways[IMAGE_scenario]["Region"].copy().replace(IMAGE_recion_code_to_iso)
df_IMAGE_pathways[IMAGE_scenarios[1]].head(25)



Units for IMAGE_ELV-SSP2-CP: Emissions: Mt CO2/yr, Population: million

Units for IMAGE_ELV-SSP2-CP: Emissions per capita: Mt CO2/yr/million

Units for IMAGE_ELV-SSP2-1150F: Emissions: Mt CO2/yr, Population: million

Units for IMAGE_ELV-SSP2-1150F: Emissions per capita: Mt CO2/yr/million


,Model,Scenario,Region,year,Unit,Variable,value,iso
0,IMAGE,ELV-SSP2-1150F,BRA,2005,Mt CO2/yr/million,emissions_per_capita,1.701322,BRA
1,IMAGE,ELV-SSP2-1150F,BRA,2010,Mt CO2/yr/million,emissions_per_capita,1.874985,BRA
2,IMAGE,ELV-SSP2-1150F,BRA,2015,Mt CO2/yr/million,emissions_per_capita,2.212527,BRA
3,IMAGE,ELV-SSP2-1150F,BRA,2020,Mt CO2/yr/million,emissions_per_capita,1.824969,BRA
4,IMAGE,ELV-SSP2-1150F,BRA,2025,Mt CO2/yr/million,emissions_per_capita,2.205315,BRA
5,IMAGE,ELV-SSP2-1150F,BRA,2030,Mt CO2/yr/million,emissions_per_capita,2.129875,BRA
6,IMAGE,ELV-SSP2-1150F,BRA,2035,Mt CO2/yr/million,emissions_per_capita,1.738825,BRA
7,IMAGE,ELV-SSP2-1150F,BRA,2040,Mt CO2/yr/million,emissions_per_capita,1.303704,BRA
8,IMAGE,ELV-SSP2-1150F,BRA,2045,Mt CO2/yr/million,emissions_per_capita,1.012487,BRA
9,IMAGE,ELV-SSP2-1150F,BRA,2050,Mt CO2/yr/million,emissions_per_capita,0.662873,BRA


## Gridded population and gridded harmonised emissions

In [23]:
# read in population and emissions
round = "second_round"
data_sources = "2UP_GHSL_2024_M3_Murakami_version_2021_1_EDGAR_2024_net"
xr_pop: dict[str, xr.Dataset] = {}
xr_em_harmonised: dict[str, xr.Dataset] = {}

for scenario in IMAGE_scenarios:
    dir_data = project_dir / f"data/processed/{round}/{data_sources}/{scenario}"

    pop_file = "Population_processed_2UP_GHSL_2024_M3_SSP2_cf_12.nc"
    xr_pop[scenario]: xr.Dataset = xr.open_dataset(dir_data / pop_file)
    arc_seconds, arc_minutes, arc_degrees = calculate_resolution(xr_pop[scenario]["Population"])
    print(f"\nPopulation dataset resolution: {arc_seconds} arc seconds, {arc_minutes} arc minutes, {arc_degrees} arc degrees")
    print(f"\nUnit: {xr_pop[scenario].attrs['unit']}")
    print(f"\n{xr_pop[scenario]}")

    em_harmonised_file = "Emissions_CO2_Excl_shipping_aviation_AFOLU_harmonised_SSP2.nc"
    xr_em_harmonised[scenario]: xr.Dataset = xr.open_dataset(dir_data / em_harmonised_file)
    arc_seconds, arc_minutes, arc_degrees = calculate_resolution(xr_em_harmonised[scenario]["Emissions_CO2_Excl_shipping_aviation_AFOLU"])
    print(f"\nEmissions dataset resolution: {arc_seconds} arc seconds, {arc_minutes} arc minutes, {arc_degrees} arc degrees")
    print(f"\nUnit: {xr_em_harmonised[scenario]['Emissions_CO2_Excl_shipping_aviation_AFOLU'].attrs['unit']}")
    print(f"\n{xr_em_harmonised[scenario]}")

2026-08-28 17:22:17 - debug - read_process_grid_data:calculate_resolution:376 (from 471650670:<module>:12) - INFO - Calculate resolution for DataArray: <xarray.DataArray 'Population' (time: 12, y: 1800, x: 3600)> Size: 622MB
[77760000 values with dtype=float64]
Coordinates:
  * x            (x) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * y            (y) float64 14kB 89.95 89.85 89.75 ... -89.75 -89.85 -89.95
  * time         (time) int64 96B 2020 2025 2030 2035 ... 2070 2080 2090 2100
    spatial_ref  int64 8B ...
Attributes:
    AREA_OR_POINT:  Area
    unit:           people
2026-08-28 17:22:17 - debug - read_process_grid_data:calculate_resolution:377 (from 471650670:<module>:12) - INFO - CRS: EPSG:4326
2026-08-28 17:22:17 - debug - read_process_grid_data:calculate_resolution:376 (from 471650670:<module>:19) - INFO - Calculate resolution for DataArray: <xarray.DataArray 'Emissions_CO2_Excl_shipping_aviation_AFOLU' (time: 12,
                                          


Population dataset resolution: 360.0 arc seconds, 6.0 arc minutes, 0.1 arc degrees

Unit: people

<xarray.Dataset> Size: 622MB
Dimensions:      (time: 12, y: 1800, x: 3600)
Coordinates:
  * x            (x) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * y            (y) float64 14kB 89.95 89.85 89.75 ... -89.75 -89.85 -89.95
  * time         (time) int64 96B 2020 2025 2030 2035 ... 2070 2080 2090 2100
    spatial_ref  int64 8B ...
Data variables:
    Population   (time, y, x) float64 622MB ...
Attributes:
    unit:     people

Emissions dataset resolution: 360.0 arc seconds, 6.0 arc minutes, 0.1 arc degrees

Unit: tonnes CO2/year

<xarray.Dataset> Size: 940MB
Dimensions:                                     (time: 12, y: 1800, x: 3600)
Coordinates:
  * x                                           (x) float64 29kB -179.9 ... 1...
  * y                                           (y) float64 14kB 89.95 ... -8...
  * time                                        (time) int64 96B 2

## Urban classification (DLL)

In [24]:
# Read in processed urban classification
dir_urban_classification = project_dir / "data/processed/DLL"
file_urban_classification = "urban_classification_years.parquet"
path_urban_classification = dir_urban_classification / file_urban_classification
gdf_urban_classification = gpd.read_parquet(path_urban_classification)
print(gdf_urban_classification.dtypes)
gdf_urban_classification.head(10)

UID                   int64
GID_2                object
Stat/province        object
County/district      object
geometry           geometry
cluster_2015        float64
cluster_2020        float64
cluster_2030        float64
cluster_2040        float64
cluster_2050        float64
cluster_2060        float64
cluster_2070        float64
cluster_2080        float64
cluster_2090        float64
cluster_2100        float64
dtype: object


,UID,GID_2,Stat/province,County/district,geometry,cluster_2015,cluster_2020,cluster_2030,cluster_2040,cluster_2050,cluster_2060,cluster_2070,cluster_2080,cluster_2090,cluster_2100
0,1,AFG.1.1_1,Badakhshan,Baharak,"MULTIPOLYGON (((71.41149 36.55717, 71.40954 36...",0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
1,2,AFG.1.2_1,Badakhshan,Darwaz,"MULTIPOLYGON (((71.2762 38.00465, 71.27578 38....",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,3,AFG.1.3_1,Badakhshan,Fayzabad,"MULTIPOLYGON (((70.78272 37.27678, 70.78635 37...",0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
3,4,AFG.1.4_1,Badakhshan,Ishkashim,"MULTIPOLYGON (((71.41149 36.55717, 71.40091 36...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
4,5,AFG.1.5_1,Badakhshan,Jurm,"MULTIPOLYGON (((70.71236 37.07621, 70.73582 37...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,6,AFG.1.6_1,Badakhshan,Khwahan,"MULTIPOLYGON (((70.41875 38.07549, 70.4374 38....",0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0
6,7,AFG.1.7_1,Badakhshan,Kishim,"MULTIPOLYGON (((70.09976 37.00258, 70.10191 36...",0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,1.0
7,8,AFG.1.8_1,Badakhshan,Kuran Wa Munjan,"MULTIPOLYGON (((70.48054 36.17657, 70.50142 36...",0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,9,AFG.1.9_1,Badakhshan,Ragh,"MULTIPOLYGON (((70.76729 37.28288, 70.74894 37...",0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
9,10,AFG.1.10_1,Badakhshan,Shahri Buzurg,"MULTIPOLYGON (((70.32571 37.46014, 70.3063 37....",0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0


In [25]:
# Urban classification: precompute urban masks for each IMAGE region
output_dir = project_dir / "data/processed/DLL"
cluster_years = [1990, 2000, 2010, 2020]

da_urban_masks: dict[str, xr.DataArray] = {}

for IMAGE_scenario in IMAGE_scenarios:
    out_path = output_dir / f"urban_masks_{IMAGE_scenario}.nc"
    print(f"\nPrecomputing urban masks for {IMAGE_scenario} and saving to {out_path} ...")
    if not out_path.exists():
        da_urban_masks[IMAGE_scenario] = precompute_urban_masks(gdf_urban_classification, xr_em_harmonised[IMAGE_scenario], out_path)
        print(f"\n{IMAGE_scenario}: {type(da_urban_masks[IMAGE_scenario])}")
        print(da_urban_masks[IMAGE_scenario])
    else:
        print(f"Urban masks for {IMAGE_scenario} already exist at {out_path}. Loading from file...")
        da_urban_masks[IMAGE_scenario] = xr.open_dataarray(out_path, decode_coords="all")
        print(f"\n{IMAGE_scenario}: {type(da_urban_masks[IMAGE_scenario])}")
        print(da_urban_masks[IMAGE_scenario])


Precomputing urban masks for IMAGE_ELV-SSP2-CP and saving to k:\PythonWork\downscaling\Kaya_downscaling\data\processed\DLL\urban_masks_IMAGE_ELV-SSP2-CP.nc ...
Urban masks for IMAGE_ELV-SSP2-CP already exist at k:\PythonWork\downscaling\Kaya_downscaling\data\processed\DLL\urban_masks_IMAGE_ELV-SSP2-CP.nc. Loading from file...

IMAGE_ELV-SSP2-CP: <class 'xarray.core.dataarray.DataArray'>
<xarray.DataArray 'is_urban' (year: 10, y: 1800, x: 3600)> Size: 65MB
[64800000 values with dtype=uint8]
Coordinates:
  * year         (year) int64 80B 2015 2020 2030 2040 ... 2070 2080 2090 2100
  * y            (y) float64 14kB 89.95 89.85 89.75 ... -89.75 -89.85 -89.95
  * x            (x) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
    spatial_ref  int64 8B ...

Precomputing urban masks for IMAGE_ELV-SSP2-1150F and saving to k:\PythonWork\downscaling\Kaya_downscaling\data\processed\DLL\urban_masks_IMAGE_ELV-SSP2-1150F.nc ...
Urban masks for IMAGE_ELV-SSP2-1150F already exist at k:\Pytho

## Pledge pathways (from paper)

In [26]:
# Selecte 2020 pledge emissions per capita for each country/region from the 'paper' dataset
dir_DLL = project_dir / "data/input/DLL"
df_em_cap_pledges = pd.read_csv(dir_DLL / "Timeseries_iso_pledge.csv", sep=",")
df_em_cap_pledges_2020 = df_em_cap_pledges[df_em_cap_pledges["year"] == 2020]
print(df_em_cap_pledges_2020["iso"].unique())
df_em_cap_pledges_2020

['ARG' 'AUS' 'BRA' 'CAN' 'CHN' 'EU' 'GBR' 'IDN' 'IND' 'JPN' 'KOR' 'MEX'
 'TUR' 'USA' 'ZAF' 'G20']


,iso,year,total_emission,total_pop,emissions_per_cap,GID_0
0,ARG,2020,1.452249e+07,4.055325e+06,3.581092,ARG
4,AUS,2020,1.533442e+07,9.351557e+05,16.397722,AUS
8,BRA,2020,3.100614e+07,2.051767e+07,1.511192,BRA
12,CAN,2020,9.560045e+07,1.147191e+07,8.333435,CAN
16,CHN,2020,5.304816e+08,6.813052e+07,7.786255,CHN
20,EU,2020,2.423387e+09,8.371751e+07,28.947188,EU
24,GBR,2020,9.027004e+07,1.645483e+07,5.485931,GBR
28,IDN,2020,1.679789e+07,2.594085e+06,6.475458,IDN
32,IND,2020,4.611363e+07,1.794940e+07,2.569091,IND
36,JPN,2020,5.290525e+08,5.100057e+07,10.373462,JPN


# Calculations and plotting

In [27]:
# calculate emissions per capita per region and country
da_em_per_capita: dict[str, xr.DataArray] = {}
ds_em_per_capita: dict[str, xr.DataArray] = {}

for scenario in IMAGE_scenarios:
    # 1. Reindex GADM regions raster to match the population dataset's coordinates using nearest neighbor interpolation
    region_on_pop = GADM_regions_raster["region_number"].reindex(x=xr_pop[scenario]["x"], y=xr_pop[scenario]["y"], method="nearest", tolerance=0.05, fill_value=-1)
    country_on_pop = GADM_regions_raster["country_id_GADM"].reindex(x=xr_pop[scenario]["x"], y=xr_pop[scenario]["y"], method="nearest", tolerance=0.05, fill_value=np.nan)
    region_on_em = GADM_regions_raster["region_number"].reindex(x=xr_em_harmonised[scenario]["x"], y=xr_em_harmonised[scenario]["y"], method="nearest", tolerance=0.05, fill_value=-1)
    country_on_em = GADM_regions_raster["country_id_GADM"].reindex(x=xr_em_harmonised[scenario]["x"], y=xr_em_harmonised[scenario]["y"], method="nearest", tolerance=0.05, fill_value=np.nan)

    # 2. Assign the reindexed region and country coordinates to the population dataset
    xr_pop[scenario] = xr_pop[scenario].assign_coords(region_number=region_on_pop.astype("int8"), country_id_GADM=country_on_pop.astype("float32"))
    print(f"\nPopulation dataset with region and country coordinates:\n{xr_pop[scenario]}")
    xr_em_harmonised[scenario] = xr_em_harmonised[scenario].assign_coords(region_number=(("y", "x"), region_on_em.astype("int8").data),country_id_GADM=(("y", "x"), country_on_em.astype("float32").data))
    print(f"\nEmissions dataset with region and country coordinates:\n{xr_em_harmonised[scenario]}")

    # 3. Calculate emissions per capita
    da_em_per_capita[scenario] = xr_em_harmonised[scenario]["Emissions_CO2_Excl_shipping_aviation_AFOLU"] / xr_pop[scenario]["Population"].where(xr_pop[scenario]["Population"] != 0)  # Avoid division by zero
    ds_em_per_capita[scenario]: xr.Dataset = da_em_per_capita[scenario].to_dataset(name=varname_em_per_capita)
    print(f"\nEmissions per capita dataset:\n{ds_em_per_capita[scenario]}")


Population dataset with region and country coordinates:
<xarray.Dataset> Size: 655MB
Dimensions:          (time: 12, y: 1800, x: 3600)
Coordinates:
  * x                (x) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
  * y                (y) float64 14kB 89.95 89.85 89.75 ... -89.75 -89.85 -89.95
  * time             (time) int64 96B 2020 2025 2030 2035 ... 2080 2090 2100
    spatial_ref      int64 8B ...
    region_number    (y, x) int8 6MB 0 0 0 0 0 0 0 0 ... 27 27 27 27 27 27 27 27
    country_id_GADM  (y, x) float32 26MB nan nan nan nan nan ... 9.0 9.0 9.0 9.0
Data variables:
    Population       (time, y, x) float64 622MB ...
Attributes:
    unit:     people

Emissions dataset with region and country coordinates:
<xarray.Dataset> Size: 966MB
Dimensions:                                     (time: 12, y: 1800, x: 3600)
Coordinates:
  * x                                           (x) float64 29kB -179.9 ... 1...
  * y                                           (y) float64

In [ ]:
# Compare
# City pledges per country
# IMAGE urban pathways per region
# IMAGE selected urban pathways per region
# IMAGE full pathways per region

# TO DO --> following steps
# 1. Make additional selection based on urban/rural pixel
#    - for IMAGE full pathways
#    - for IMAGE selected pathways
# 2. Include countries in addition to regions

# read IAM IMAGE translation region numbers to ISO codes
import time

ISO_codes_plot =['ARG', 'AUS', 'BRA', 'CAN', 'CHN', 'EU', 'GBR', 'IDN', 'IND', 'JPN', 'KOR','MEX', 'TUR', 'USA', 'ZAF', 'G20']
#ISO_selection =['BRA', 'CAN', 'CHN', 'IDN', 'IND', 'JPN', 'KOR', 'MEX', 'TUR', 'USA', 'ZAF']
ISO_selection =['CAN', 'IDN', 'JPN']

df_IMAGE_pathways_region: dict[str, pd.DataFrame] = {}
df_IMAGE_urban_pathways: dict[str, pd.DataFrame] = {}
df_IMAGE_urban_pathways_region: dict[str, pd.DataFrame] = {}
df_region_totals: dict[str, pd.DataFrame] = {}
df_urban_region_totals: dict[str, pd.DataFrame] = {}
ds_em_per_capita_selected: dict[str, xr.Dataset] = {}
ds_urban_em_per_capita_selected: dict[str, xr.Dataset] = {}

for iso in ISO_selection:
    startime = time.time()
    region_number = iso_to_region.get(iso)
    if region_number is not None:
        region_number = int(region_number)
    print(f"\n{"\033[92m"}Processing ISO code: {iso}, IMAGE region number: {region_number}{"\033[0m"}")

    per_capita_threshold = float(df_em_cap_pledges_2020[df_em_cap_pledges_2020["iso"]==iso]["emissions_per_cap"].values[0])
    print(type(per_capita_threshold))
    print(f"Per capita threshold: {per_capita_threshold:.2f}")

    print(f"Processing time: {time.time() - startime:.2f} seconds")

    # Retrieve or calculate pahtways
    print(f"Retrieving IMAGE pathways for region {region_number} (ISO: {iso})...")
    df_pledge_cities_country = df_em_cap_pledges[df_em_cap_pledges["iso"] == iso]

    print(f"Processing time: {time.time() - startime:.2f} seconds")
    # Select the IMAGE pathways for the specific region
    print(f"Selecting IMAGE pathways for region {region_number} (ISO: {iso})...")
    for IMAGE_scenario in IMAGE_scenarios:
        df_IMAGE_pathways_region[IMAGE_scenario] = df_IMAGE_pathways[IMAGE_scenario][df_IMAGE_pathways[IMAGE_scenario]["iso"] == iso]
        df_IMAGE_pathways_region[IMAGE_scenario].to_csv(project_dir / "data/check/urban_comparison" / f"df_IMAGE_pathways_region_{iso}.csv", index=False, sep=";")
    print(f"Processing time: {time.time() - startime:.2f} seconds")

    # Calculate IMAGE urban pathways for the specific region
    print(f"Calculating IMAGE urban pathways for region {region_number} (ISO: {iso})...")
    for scenario, IMAGE_scenario in zip(scenarios, IMAGE_scenarios):
        if calculate_IMAGE_urban_pathways:
            with contextlib.redirect_stdout(io.StringIO()): # disable output from aggregate_urban_values
                dummy1, df_IMAGE_urban_pathways_region[IMAGE_scenario], dummy2 = aggregate_urban_values(xr_em_harmonised[IMAGE_scenario], gdf_urban_classification, varname_em, "region_number", 2050)
        else:
            df_IMAGE_urban_pathways[IMAGE_scenario] = pd.read_csv(project_dir / "data/output/" / f"Emissions_combined_region_{scenario}_{"second_round"}_harmonised.csv", sep=";")
        df_IMAGE_urban_pathways[IMAGE_scenario].drop(["rural", "total"], axis=1, inplace=True)
        df_IMAGE_urban_pathways[IMAGE_scenario].rename(columns={"urban": "value"}, inplace=True)
        df_IMAGE_urban_pathways_region[IMAGE_scenario] = df_IMAGE_urban_pathways[IMAGE_scenario][df_IMAGE_urban_pathways[IMAGE_scenario]["region_number"] == region_number]
        df_IMAGE_urban_pathways_region[IMAGE_scenario].to_csv(project_dir / "data/check/urban_comparison" / f"df_IMAGE_urban_pathways_region_{iso}.csv", index=False, sep=";")
        #print(df_IMAGE_urban_pathways_region[IMAGE_scenario].head(10))
    print(f"Processing time: {time.time() - startime:.2f} seconds")

    # Select cells based on per capita emissions for each scenario
    print(f"Selecting cells for {IMAGE_scenarios[0]} based on per capita emissions for region {region_number} (ISO: {iso})...")
    df_region_totals[IMAGE_scenarios[0]], ds_em_per_capita_selected[IMAGE_scenarios[0]] = select_cells_based_on_per_capita(region_number,
                                                                                   ds_em_per_capita[IMAGE_scenarios[0]], varname_em_per_capita,
                                                                                   xr_pop[IMAGE_scenarios[0]], "Population",
                                                                                   only_urban=False,
                                                                                   da_urban_masks=da_urban_masks[IMAGE_scenarios[0]],
                                                                                   per_capita_threshold=per_capita_threshold,
                                                                                   perc_select=0.05,
                                                                                   arc_minutes=6.00)
    print(f"Processing time: {time.time() - startime:.2f} seconds")
    print(f"Selecting cells for {IMAGE_scenarios[1]} based on per capita emissions for region {region_number} (ISO: {iso})...")
    df_region_totals[IMAGE_scenarios[1]], ds_em_per_capita_selected[IMAGE_scenarios[1]] = select_cells_based_on_per_capita(region_number,
                                                                                   ds_em_per_capita[IMAGE_scenarios[1]], varname_em_per_capita,
                                                                                   xr_pop[IMAGE_scenarios[1]], "Population",
                                                                                   only_urban=False,
                                                                                   da_urban_masks=da_urban_masks[IMAGE_scenarios[1]],
                                                                                   per_capita_threshold=per_capita_threshold,
                                                                                   perc_select=0.05,
                                                                                   arc_minutes=6.00)

    print(f"Type of ds_urban_em_per_capita_selected[IMAGE_scenarios[0]]: {type(ds_em_per_capita_selected[IMAGE_scenarios[0]])}")
    print(f"Processing time: {time.time() - startime:.2f} seconds")
    # # Select urban cells based on per capita emissions for each scenario
    # print(f"Selecting urban cells for {IMAGE_scenarios[0]} based on per capita emissions for region {region_number} (ISO: {iso})...")
    # df_urban_region_totals[IMAGE_scenarios[0]], ds_urban_em_per_capita_selected[IMAGE_scenarios[0]] = select_cells_based_on_per_capita(region_number,
    #                                                                                                                                     ds_em_per_capita[IMAGE_scenarios[0]], varname_em_per_capita,
    #                                                                                                                                     xr_pop[IMAGE_scenarios[0]], "Population",
    #                                                                                                                                     only_urban=True,
    #                                                                                                                                     da_urban_masks=da_urban_masks[IMAGE_scenarios[0]],
    #                                                                                                                                     per_capita_threshold=per_capita_threshold,
    #                                                                                                                                     perc_select=0.05,
    #                                                                                                                                     arc_minutes=6.00)
    # print(f"Processing time: {time.time() - startime:.2f} seconds")
    # print(f"Selecting urban cells for {IMAGE_scenarios[1]} based on per capita emissions for region {region_number} (ISO: {iso})...")
    # df_urban_region_totals[IMAGE_scenarios[1]], ds_urban_em_per_capita_selected[IMAGE_scenarios[1]] = select_cells_based_on_per_capita(region_number,
    #                                                                                                                                     ds_em_per_capita[IMAGE_scenarios[1]], varname_em_per_capita,
    #                                                                                                                                     xr_pop[IMAGE_scenarios[1]], "Population",
    #                                                                                                                                     only_urban=True,
    #                                                                                                                                     da_urban_masks=da_urban_masks[IMAGE_scenarios[1]],
    #                                                                                                                                     per_capita_threshold=per_capita_threshold,
    #                                                                                                                                     perc_select=0.05,
    #                                                                                                                                     arc_minutes=6.00)
    # print(f"Type of ds_urban_em_per_capita_selected[IMAGE_scenarios[0]]: {type(ds_urban_em_per_capita_selected[IMAGE_scenarios[0]])}")
    # print(f"Type of ds_urban_em_per_capita_selected[IMAGE_scenarios[1]]: {type(ds_urban_em_per_capita_selected[IMAGE_scenarios[1]])}")
    # print(f"Processing time: {time.time() - startime:.2f} seconds")
    _plot_pathways(project_dir, region_number, ds_em_per_capita_selected, ds_urban_em_per_capita_selected, varname_em_per_capita,
                   df_IMAGE_pathways_region, df_IMAGE_urban_pathways_region,
                   df_pledge_cities_country)

    print(f"Total processing time: {time.time() - startime:.2f} seconds")


Region to ISO mapping:
{1: 'CAN', 2: 'USA', 3: 'MEX', 4: 'RCAM', 5: 'BRA', 6: 'RSAM', 7: 'NAF', 8: 'WAF', 9: 'EAF', 10: 'ZAF', 11: 'WEU', 12: 'CEU', 13: 'TUR', 14: 'UKR', 15: 'STAN', 16: 'RUS', 17: 'ME', 18: 'IND', 19: 'KOR', 20: 'CHN', 21: 'SEAS', 22: 'IDN', 23: 'JPN', 24: 'OCE', 25: 'RSAS', 26: 'RSAF', 27: 'ATA'}

ISO to Region mapping:
{'CAN': 1, 'USA': 2, 'MEX': 3, 'RCAM': 4, 'BRA': 5, 'RSAM': 6, 'NAF': 7, 'WAF': 8, 'EAF': 9, 'ZAF': 10, 'WEU': 11, 'CEU': 12, 'TUR': 13, 'UKR': 14, 'STAN': 15, 'RUS': 16, 'ME': 17, 'IND': 18, 'KOR': 19, 'CHN': 20, 'SEAS': 21, 'IDN': 22, 'JPN': 23, 'OCE': 24, 'RSAS': 25, 'RSAF': 26, 'ATA': 27}

Processing ISO code: CAN, IMAGE region number: 1
<class 'float'>
Per capita threshold: 8.33
Processing time: 0.00 seconds
Retrieving IMAGE pathways for region 1 (ISO: CAN)...
Processing time: 0.00 seconds
Selecting IMAGE pathways for region 1 (ISO: CAN)...
Processing time: 0.01 seconds
Calculating IMAGE urban pathways for region 1 (ISO: CAN)...
Processing time: